## Cellule 1 : Imports et chargement

In [103]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

df = pd.read_csv("...data/marketing_campaign.csv", sep = None)  # sep=None détecte ; ou \t

print(df.shape)
df.head()

C:\Users\Olouwafemi\AppData\Local\Temp\ipykernel_15384\2835991477.py:7: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support sep=None with delim_whitespace=False; you can avoid this warning by specifying engine='python'.
  df = pd.read_csv("F:/Téléchargements/data/marketing_campaign.csv", sep = None)  # sep=None détecte ; ou \t


(2240, 29)


,﻿ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


## Cellule 2 : Aperçu de la structure

In [105]:
# Nettoyage des noms de colonnes (espaces parasites)
df.columns = df.columns.str.strip()

df.info()
print("\nValeurs manquantes :")
print(df.isna().sum()[df.isna().sum() > 0])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ﻿ID                  2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

## Cellule 3 : Doublons exacts


In [107]:
n_dup = df.duplicated().sum()
print(f"Doublons exacts : {n_dup}")

if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print(f"Dimensions après suppression : {df.shape}")

Doublons exacts : 0
Dimensions après suppression : (2240, 29)


## Cellule 4 : Correction des types

In [109]:
# --- Income : retirer symboles monétaires / séparateurs si c'est du texte
if df["Income"].dtype == "object":
    df["Income"] = (df["Income"].astype(str)
                    .str.replace(r"[^\d.\-]", "", regex=True)
                    .replace("", np.nan))
df["Income"] = pd.to_numeric(df["Income"], errors="coerce")

# --- Variables numériques
num_cols = [
    "Income", "Kidhome", "Teenhome", "Recency",
    "MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts",
    "MntSweetProducts", "MntGoldProds",
    "NumDealsPurchases", "NumWebPurchases", "NumCatalogPurchases",
    "NumStorePurchases", "NumWebVisitsMonth",
]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# --- Variables binaires -> entiers
bin_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
            "AcceptedCmp4", "AcceptedCmp5", "Response", "Complain"]
df[bin_cols] = df[bin_cols].apply(pd.to_numeric, errors="coerce")

df.dtypes

﻿ID                      int64
Year_Birth               int64
Education               object
Marital_Status          object
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer             object
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Response                 int64
dtype: object

## Cellule 5 : Anomalies métier

In [111]:
# 5.1 Valeurs négatives (montants, comptages, récence, revenu)
non_neg_cols = [c for c in num_cols]
neg_report = (df[non_neg_cols] < 0).sum()
print("Valeurs négatives par colonne :")
print(neg_report[neg_report > 0] if neg_report.sum() else "Aucune")

# Décision : une dépense/quantité négative n'a pas de sens ici (pas de retours
# dans ce dataset) -> on la considère invalide (NaN) plutôt que de supprimer la ligne
for c in non_neg_cols:
    df.loc[df[c] < 0, c] = np.nan

# 5.2 Variables binaires : valeurs hors {0, 1}
for c in bin_cols:
    invalid = ~df[c].isin([0, 1]) & df[c].notna()
    if invalid.any():
        print(f"{c} : {invalid.sum()} valeurs invalides -> NaN")
        df.loc[invalid, c] = np.nan

# 5.3 Revenus nuls ou aberrants (ex. > 99,9e percentile très éloigné)
print("\nRevenus <= 0 :", (df["Income"] <= 0).sum())
df.loc[df["Income"] <= 0, "Income"] = np.nan

q_low, q_high = df["Income"].quantile([0.01, 0.999])
print(f"Income : P1 = {q_low:,.0f} | P99.9 = {q_high:,.0f} | max = {df['Income'].max():,.0f}")
df["Income_outlier"] = (df["Income"] > q_high).astype(int)  # on signale, on ne supprime pas



# 5.5 Valeurs manquantes : on ne supprime que les lignes sans information clé
print("\nValeurs manquantes restantes :")
print(df.isna().sum()[df.isna().sum() > 0])
df = df.dropna(subset=["Response"]).reset_index(drop=True)  # cible obligatoire
# Income : l'imputation sera traitée dans l'étape de feature engineering / modélisation

# Les colonnes binaires/catégorielles sans NaN peuvent redevenir entières
df[bin_cols] = df[bin_cols].astype("int64")

Valeurs négatives par colonne :
Aucune

Revenus <= 0 : 0
Income : P1 = 7,579 | P99.9 = 160,143 | max = 666,666

Valeurs manquantes restantes :
Income    24
dtype: int64


## Cellule 6 : Standardisation des colonnes texte

In [113]:
text_cols = ["Education", "Marital_Status"]
for c in text_cols:
    df[c] = (df[c].astype(str).str.strip()
             .str.replace(r"\s+", " ", regex=True)
             .str.title())

print(df["Education"].value_counts(), "\n")
print(df["Marital_Status"].value_counts())

Education
Graduation    1127
Phd            486
Master         370
2N Cycle       203
Basic           54
 ame: count, dtype: int64

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
Yolo          2
Name: count, dtype: int64


In [114]:
# Regroupement des modalités équivalentes ou incohérentes
education_map = {
    "2N Cycle": "Master",
    "2n Cycle": "Master",
    "Graduation": "Graduate",
    "Basic": "Basic",
    "Master": "Master",
    "Phd": "PhD",
}
marital_map = {
    "Alone": "Single",
    "Absurd": "Other",
    "Yolo": "Other",
    "Together": "Together",
    "Married": "Married",
    "Divorced": "Divorced",
    "Widow": "Widow",
    "Single": "Single",
}

df["Education"] = df["Education"].replace(education_map)
df["Marital_Status"] = df["Marital_Status"].replace(marital_map)

# Conversion en type catégoriel
df[["Education", "Marital_Status"]] = df[["Education", "Marital_Status"]].astype("category")

print(df["Education"].value_counts(), "\n")
print(df["Marital_Status"].value_counts())

Education
Graduate    1127
Master       573
PhD          486
Basic         54
Name: count, dtype: int64 

Marital_Status
Married     864
Together    580
Single      483
Divorced    232
Widow        77
Other         4
Name: count, dtype: int64


## Cellule 7 : Vérifications finales et sauvegarde

In [116]:
print("Dimensions finales :", df.shape)
print("Doublons restants  :", df.duplicated().sum())
df.info()

Dimensions finales : (2240, 30)
Doublons restants  : 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   ﻿ID                  2240 non-null   int64   
 1   Year_Birth           2240 non-null   int64   
 2   Education            2240 non-null   category
 3   Marital_Status       2240 non-null   category
 4   Income               2216 non-null   float64 
 5   Kidhome              2240 non-null   float64 
 6   Teenhome             2240 non-null   float64 
 7   Dt_Customer          2240 non-null   object  
 8   Recency              2240 non-null   float64 
 9   MntWines             2240 non-null   float64 
 10  MntFruits            2240 non-null   float64 
 11  MntMeatProducts      2240 non-null   float64 
 12  MntFishProducts      2240 non-null   float64 
 13  MntSweetProducts     2240 non-null   float64 
 14  MntGoldProds     

In [117]:
Path("data").mkdir(exist_ok=True)
df.to_csv("data/marketing_campaign_clean.csv", index=False)
print("Fichier sauvegardé : data/marketing_campaign_clean.csv")

Fichier sauvegardé : data/marketing_campaign_clean.csv
